# 🤖 NLP Disaster Tweets — RoBERTa K-Fold Training

> **Kaggle Competition**: [Natural Language Processing with Disaster Tweets](https://www.kaggle.com/c/nlp-getting-started)  
> **목표**: RoBERTa를 K-Fold Cross-Validation으로 학습 후, Fold 앙상블로 테스트셋 예측

## 🏗️ 파이프라인 구조
```
데이터 로드 & 전처리
    ↓
Tweet 정규화 (normalizeTweet)
    ↓
K-Fold 학습 (FGM Adversarial Training + Cosine LR Schedule)
    ↓
Fold별 Best Model 저장
    ↓
앙상블 추론 (Soft Voting) → submission.csv
```

## 📋 목차
1. [환경 설정](#1)
2. [데이터 전처리](#2)
3. [모델 컴포넌트 정의](#3)
4. [K-Fold 학습 실행](#4)
5. [앙상블 추론 & 제출](#5)


## 1. 환경 설정 <a id='1'></a>

In [ ]:
# 필요 패키지 설치 (Colab 환경)
!pip install transformers datasets emoji -q


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import (
    RobertaTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from tqdm import tqdm
from emoji import demojize
from nltk.tokenize import TweetTokenizer


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/JM/ai-스터디/transformer/nlp-Disaster_Tweets-kaggle'
os.chdir(BASE_PATH)
print('작업 경로:', os.getcwd())


In [ ]:
def set_seed(seed: int = 42) -> None:
    """재현성을 위해 모든 랜덤 시드를 고정합니다."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## 2. 데이터 전처리 <a id='2'></a>

### 2.1 Tweet 정규화
트위터 텍스트의 특성을 고려한 정규화를 수행합니다.
- `@mention` → `@USER`
- URL → `HTTPURL`
- 이모지 → 텍스트 변환 (`demojize`)
- 축약형 분리 (`can't` → `can not`)


In [ ]:
_tweet_tokenizer = TweetTokenizer()


def _normalize_token(token: str) -> str:
    lower = token.lower()
    if token.startswith('@'):
        return '@USER'
    elif lower.startswith('http') or lower.startswith('www'):
        return 'HTTPURL'
    elif len(token) == 1:
        return demojize(token)
    else:
        return token.replace("'", "'").replace('…', '...')


def normalize_tweet(tweet: str) -> str:
    """트위터 텍스트를 RoBERTa 입력에 맞게 정규화합니다."""
    tokens = _tweet_tokenizer.tokenize(tweet.replace("'", "'").replace('…', '...'))
    norm = ' '.join(_normalize_token(t) for t in tokens)
    norm = (norm
        .replace('cannot ', 'can not ').replace("n't ", " n't ")
        .replace("n 't ", " n't ").replace("ca n't", "can't").replace("ai n't", "ain't")
        .replace("'m ", " 'm ").replace("'re ", " 're ").replace("'s ", " 's ")
        .replace("'ll ", " 'll ").replace("'d ", " 'd ").replace("'ve ", " 've ")
        .replace(' p . m .', '  p.m.').replace(' p . m ', ' p.m ')
        .replace(' a . m .', ' a.m.').replace(' a . m ', ' a.m ')
    )
    return ' '.join(norm.split())


In [ ]:
# 데이터 로드
train_df = pd.read_csv(BASE_PATH + '/nlp-getting-started/augmented_train_aust_easy_v2.csv')
test_df  = pd.read_csv(BASE_PATH + '/nlp-getting-started/test.csv')

# 결측치 처리 & 타입 통일
train_df['text'] = train_df['text'].fillna('').astype(str)
test_df['text']  = test_df['text'].fillna('').astype(str)
train_df = train_df.dropna(subset=['text']).reset_index(drop=True)

# Tweet 정규화 적용
train_df['text'] = train_df['text'].apply(normalize_tweet)
test_df['text']  = test_df['text'].apply(normalize_tweet)

print(f'Train: {len(train_df):,}건 | Test: {len(test_df):,}건')
print(f'Target 분포:\n{train_df["target"].value_counts(normalize=True).round(3)}')


## 3. 모델 컴포넌트 정의 <a id='3'></a>

| 컴포넌트 | 역할 |
|----------|------|
| `TweetDataset` | 텍스트 토크나이징 & DataLoader 준비 |
| `FGM` | Fast Gradient Method — 임베딩에 노이즈를 추가하는 Adversarial Training |
| `train_one_epoch` | 1 epoch 학습 (FGM 옵션) |
| `evaluate` | 검증 손실 & F1 계산 |
| `run_kfold` | K-Fold 전체 학습 루프 |


In [ ]:
class TweetDataset(Dataset):
    """RoBERTa 입력 형식으로 트윗을 토크나이징하는 Dataset."""

    def __init__(self, texts: list, labels: list = None, tokenizer=None, max_len: int = 128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=False,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


In [ ]:
class FGM:
    """
    Fast Gradient Method (FGM) Adversarial Training.
    임베딩 레이어에 노이즈를 추가해 모델의 robustness를 높입니다.
    
    Reference: Miyato et al., 'Adversarial Training Methods for Semi-supervised Text Classification'
    """

    def __init__(self, model: nn.Module, epsilon: float = 1.0):
        self.model   = model
        self.epsilon = epsilon
        self.backup  = {}

    def attack(self, emb_name: str = 'word_embeddings') -> None:
        """임베딩에 gradient 방향의 perturbation을 추가합니다."""
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                if param.grad is None:
                    continue
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0:
                    param.data.add_(self.epsilon * param.grad / norm)

    def restore(self, emb_name: str = 'word_embeddings') -> None:
        """임베딩을 원래 값으로 복원합니다."""
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


In [ ]:
def train_one_epoch(
    model, train_loader, optimizer, scheduler, device, use_fgm: bool = True
) -> tuple[float, float]:
    """1 epoch 학습. FGM adversarial training 옵션 포함."""
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    fgm = FGM(model)

    for batch in tqdm(train_loader, desc='Training', leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        outputs = model(**batch)
        outputs.loss.backward()

        if use_fgm:
            fgm.attack()
            model(**batch).loss.backward()
            fgm.restore()

        optimizer.step()
        scheduler.step()
        total_loss += outputs.loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

    return total_loss / len(train_loader), f1_score(all_labels, all_preds)


def evaluate(model, val_loader, device) -> tuple[float, float]:
    """검증 데이터에 대한 loss와 F1을 반환합니다."""
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validation', leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['labels'].cpu().numpy())

    return total_loss / len(val_loader), f1_score(all_labels, all_preds)


In [ ]:
def run_kfold(
    train_df: pd.DataFrame,
    tokenizer,
    device,
    n_splits:        int   = 3,
    epochs:          int   = 4,
    batch_size:      int   = 16,
    max_len:         int   = 128,
    lr:              float = 2e-5,
    use_fgm:         bool  = True,
    patience:        int   = 2,
    save_dir:        str   = 'exp/kfold',
) -> tuple[pd.DataFrame, list]:
    """
    Stratified K-Fold Cross-Validation으로 RoBERTa를 학습합니다.
    - Early Stopping (val_loss 기준)
    - Cosine LR Schedule with Warmup
    - FGM Adversarial Training (옵션)
    """
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    all_histories, fold_best_f1s = [], []

    for fold, (train_idx, val_idx) in enumerate(kf.split(train_df, train_df['target'])):
        print(f"\n{'='*55}")
        print(f"  Fold {fold+1}/{n_splits}")
        print(f"{'='*55}")

        fold_train = train_df.iloc[train_idx].reset_index(drop=True)
        fold_val   = train_df.iloc[val_idx].reset_index(drop=True)

        train_loader = DataLoader(
            TweetDataset(fold_train['text'].tolist(), fold_train['target'].tolist(), tokenizer, max_len),
            batch_size=batch_size, shuffle=True
        )
        val_loader = DataLoader(
            TweetDataset(fold_val['text'].tolist(), fold_val['target'].tolist(), tokenizer, max_len),
            batch_size=batch_size, shuffle=False
        )

        model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
        model.to(device)
        optimizer = AdamW(model.parameters(), lr=lr)

        total_steps  = len(train_loader) * epochs
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
        )

        save_path = Path(save_dir) / f'fold{fold+1}'
        save_path.mkdir(parents=True, exist_ok=True)

        best_val_loss, best_val_f1, early_stop_counter = float('inf'), 0.0, 0
        history = []

        for epoch in range(epochs):
            print(f"\n  Epoch {epoch+1}/{epochs}")
            train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, scheduler, device, use_fgm)
            val_loss,   val_f1   = evaluate(model, val_loader, device)
            print(f"  Train → Loss: {train_loss:.4f} | F1: {train_f1:.4f}")
            print(f"  Val   → Loss: {val_loss:.4f}   | F1: {val_f1:.4f}")

            is_best = val_loss < best_val_loss
            if is_best:
                best_val_loss, best_val_f1 = val_loss, val_f1
                early_stop_counter = 0
                torch.save(model.state_dict(), save_path / 'best_model.pt')
                print(f"  ✅ Best model saved (val_loss={best_val_loss:.4f})")
            else:
                early_stop_counter += 1
                print(f"  No improvement ({early_stop_counter}/{patience})")
                if early_stop_counter >= patience:
                    print(f"  ⏹ Early stopping at epoch {epoch+1}")
                    break

            history.append(dict(fold=fold+1, epoch=epoch+1,
                                train_loss=train_loss, train_f1=train_f1,
                                val_loss=val_loss, val_f1=val_f1,
                                best_val_loss=best_val_loss, best_val_f1=best_val_f1, is_best=is_best))

        fold_best_f1s.append(best_val_f1)
        all_histories.append(pd.DataFrame(history))
        print(f"\n  Fold {fold+1} 완료 → Best Val Loss: {best_val_loss:.4f} | Best Val F1: {best_val_f1:.4f}")

        del model
        torch.cuda.empty_cache()

    print(f"\n{'='*55}")
    print(f"  CV 결과 요약")
    print(f"{'='*55}")
    for i, f1 in enumerate(fold_best_f1s):
        print(f"  Fold {i+1}: F1 = {f1:.4f}")
    print(f"  Mean F1: {np.mean(fold_best_f1s):.4f} ± {np.std(fold_best_f1s):.4f}")

    all_history_df = pd.concat(all_histories).reset_index(drop=True)
    all_history_df.to_csv(f'{save_dir}/cv_history.csv', index=False)
    return all_history_df, fold_best_f1s


## 4. K-Fold 학습 실행 <a id='4'></a>

| 하이퍼파라미터 | 값 |
|----------------|----|
| Model | `roberta-base` |
| Folds | 3 |
| Epochs | 4 (Early Stopping patience=2) |
| Batch size | 16 |
| Learning rate | 2e-5 |
| LR Schedule | Cosine with Warmup (10%) |
| Adversarial | FGM OFF (실험 결과 효과 없음) |


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

SAVE_DIR = 'exp/exp14_kfold_roberta_pseudo_mc_v2'

all_history_df, fold_best_f1s = run_kfold(
    train_df   = train_df,
    tokenizer  = tokenizer,
    device     = DEVICE,
    n_splits   = 3,
    epochs     = 4,
    batch_size = 16,
    lr         = 2e-5,
    use_fgm    = False,
    patience   = 2,
    save_dir   = SAVE_DIR,
)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('K-Fold Training History', fontsize=13, fontweight='bold')

for fold_id, grp in all_history_df.groupby('fold'):
    axes[0].plot(grp['epoch'], grp['train_loss'], label=f'Fold {fold_id} Train', linestyle='--', alpha=0.6)
    axes[0].plot(grp['epoch'], grp['val_loss'],   label=f'Fold {fold_id} Val')
    axes[1].plot(grp['epoch'], grp['train_f1'],   label=f'Fold {fold_id} Train', linestyle='--', alpha=0.6)
    axes[1].plot(grp['epoch'], grp['val_f1'],     label=f'Fold {fold_id} Val')

axes[0].set_title('Loss per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('F1 Score per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 5. 앙상블 추론 & 제출 <a id='5'></a>

각 Fold의 Best Model로 테스트셋을 추론한 후, **Soft Voting(확률 평균)**으로 최종 예측을 생성합니다.


In [ ]:
# Fold별 결과 요약 출력
all_history_df = pd.read_csv(f'{SAVE_DIR}/cv_history.csv')

best_per_fold = all_history_df[all_history_df['is_best']].groupby('fold').last()
print('===== Fold별 Best Validation 결과 =====')
for fold, row in best_per_fold.iterrows():
    print(f'  Fold {fold} | Val Loss: {row["val_loss"]:.4f} | Val F1: {row["val_f1"]:.4f}')
print(f"\n  Mean Val Loss : {best_per_fold['val_loss'].mean():.4f} ± {best_per_fold['val_loss'].std():.4f}")
print(f"  Mean Val F1   : {best_per_fold['val_f1'].mean():.4f} ± {best_per_fold['val_f1'].std():.4f}")


In [ ]:
test_loader = DataLoader(
    TweetDataset(test_df['text'].tolist(), tokenizer=tokenizer),
    batch_size=32, shuffle=False
)

N_SPLITS  = 3
all_probs = []

for fold in range(N_SPLITS):
    model_path = f'{SAVE_DIR}/fold{fold+1}/best_model.pt'
    model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE).eval()
    print(f'Fold {fold+1} 모델 로드 완료')

    fold_probs = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Fold {fold+1} Inference'):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            probs = torch.softmax(
                model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=1
            )
            fold_probs.extend(probs.cpu().numpy())

    all_probs.append(fold_probs)
    del model
    torch.cuda.empty_cache()

# Soft Voting 앙상블
mean_probs  = np.array(all_probs).mean(axis=0)   # (n_samples, 2)
final_preds = np.argmax(mean_probs, axis=1)

submission = pd.DataFrame({'id': test_df['id'], 'target': final_preds})
submission.to_csv(f'{SAVE_DIR}/submission_ensemble.csv', index=False)

print('\n✅ submission_ensemble.csv 저장 완료')
print(submission['target'].value_counts().rename('count').to_frame())
